# LAB EXPERIMENT – 7
Developing and Deploying APIs for ML Models



In [2]:
# step 1
!pip install -q fastapi uvicorn scikit-learn pandas joblib pyngrok nest_asyncio requests

In [3]:
#Step 2 – Train ML model and save it
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
# Evaluate
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
# Save model
joblib.dump(model, "model.pkl")
print("model.pkl saved successfully")


Accuracy: 1.0
model.pkl saved successfully


In [4]:
# Step 3 – Create FastAPI application
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
# Load trained model
model = joblib.load("model.pkl")
# Create FastAPI app
app = FastAPI(title="Iris Prediction API")
# Input schema
class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float
# Home endpoint
@app.get("/")
def home():
    return {"message": "Iris Prediction API is running"}
# Prediction endpoint
@app.post("/predict")
def predict(features: IrisFeatures):
    data = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])
    prediction = model.predict(data)[0]
    species = ["setosa", "versicolor", "virginica"][prediction]
    return {
        "prediction": int(prediction),
        "species": species
    }


In [5]:
# Step 5: Replace PASTE_YOUR_NGROK_TOKEN_HERE with your copied token.
from pyngrok import ngrok
# Add your ngrok auth token here
ngrok.set_auth_token("3JAqRcjL43Dx394OJeziT7i6Vrn_2zxxLFjLXYsqhYjz77gGZ")
print("ngrok authenticated successfully")


ngrok authenticated successfully


In [6]:
# Step 6: Run the server
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print("Public URL:", public_url.public_url)


Public URL: https://affidavit-passerby-shifter.ngrok-free.dev


In [16]:
# Step 7: Test the API
import requests
# Add /predict at the end
url = "https://affidavit-passerby-shifter.ngrok-free.dev/predict"
sample = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}
response = requests.post(url, json=sample)
print("Status Code:", response.status_code)
print("Response:", response.json())


INFO:     34.142.232.231:0 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 0, 'species': 'setosa'}


In [18]:
# Step 8: Open Swagger UI
print("Open this URL in browser:")
print("https://affidavit-passerby-shifter.ngrok-free.dev/docs")


Open this URL in browser:
https://affidavit-passerby-shifter.ngrok-free.dev/docs


In [19]:
# Step 9: Save the API code as app.py
app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
model = joblib.load("model.pkl")
app = FastAPI()
class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float
@app.get("/")
def home():
    return {"message": "API Running"}
@app.post("/predict")
def predict(features: IrisFeatures):
    data = np.array([[
        features.sepal_length,
        features.sepal_width,
        features.petal_length,
        features.petal_width
    ]])
    pred = model.predict(data)[0]
    species = ["setosa", "versicolor", "virginica"][pred]
    return {
        "prediction": int(pred),
        "species": species
    }
'''
with open("app.py", "w") as f:
    f.write(app_code)
print("app.py created successfully")


app.py created successfully


In [20]:
# Step 10: Create Dockerfile
dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY model.pkl .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
with open("Dockerfile", "w") as f:
    f.write(dockerfile)
requirements = '''
fastapi
uvicorn
scikit-learn
numpy
joblib
pydantic
'''

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("Dockerfile and requirements.txt created")


Dockerfile and requirements.txt created


In [21]:
# Step 11: Create Kubernetes deployment file
k8s_yaml = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api
spec:
  replicas: 1
  selector:
    matchLabels:
      app: iris-api
  template:
    metadata:
      labels:
        app: iris-api
    spec:
      containers:
      - name: iris-api
        image: iris-api:latest
        ports:
        - containerPort: 8000
---
apiVersion: v1
kind: Service
metadata:
  name: iris-api-service
spec:
  selector:
    app: iris-api
  ports:
  - protocol: TCP
    port: 80
    targetPort: 8000
  type: LoadBalancer
'''
with open("deployment.yaml", "w") as f:
    f.write(k8s_yaml)
print("deployment.yaml created successfully")


deployment.yaml created successfully


In [22]:
# Step 11: Download all files
from google.colab import files
files.download("app.py")
files.download("model.pkl")
files.download("Dockerfile")
files.download("requirements.txt")
files.download("deployment.yaml")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import requests
import time
# Your ngrok API URL
url = "https://affidavit-passerby-shifter.ngrok-free.dev/predict"

# Sample data
sample = {
    "sepal_length": 6.2,
    "sepal_width": 3.4,
    "petal_length": 5.4,
    "petal_width": 2.3
}

# Send request
response = requests.post(url, json=sample)

print("Status Code:", response.status_code)
print("Response:", response.json())

# Wait a moment so ngrok records it
time.sleep(2)

INFO:     34.142.232.231:0 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
Response: {'prediction': 2, 'species': 'virginica'}


In [31]:
import json

# Assuming 'sample', 'url', and 'response' are available from previous cell execution (YamClz1zj92K)

# Constructing data dictionaries to reflect the information requested
request_data = {
    "method": "POST", # As requests.post was used in the previous cell
    "uri": url,
    "json_payload": sample # The payload sent in the request
}

response_data = {
    "status_code": response.status_code,
    "json_response": response.json() # The JSON response content
}

print(json.dumps({
    "method": request_data.get("method"),
    "uri": request_data.get("uri"),
    "remote_addr": "Not available", # This information is not directly available from the client-side 'requests' object
    "status_code": response_data.get("status_code")
}, indent=2))

{
  "method": "POST",
  "uri": "https://affidavit-passerby-shifter.ngrok-free.dev/predict",
  "remote_addr": "Not available",
  "status_code": 200
}
